# Step 5 — SQL Analysis Layer
Load enriched data into SQLite and run analytical queries.

In [1]:
import pandas as pd
import sqlite3

df = pd.read_csv('../data/ai_datacenter_enriched.csv')

conn = sqlite3.connect('../data/datacenter.db')
df.to_sql('resources', conn, if_exists='replace', index=False)
print(f"Loaded {len(df)} rows into SQLite")

Loaded 50 rows into SQLite


## Query 1: Water Intensity Ranking

In [2]:
query_water = """
SELECT company, 
       ROUND(AVG(wue), 3) AS avg_wue,
       ROUND(AVG(water_per_million_queries), 0) AS avg_litres_per_M_queries,
       ROUND(SUM(water_consumed_litres) / 1e9, 2) AS total_water_billion_litres
FROM resources
WHERE year >= 2020 AND company != 'All'
GROUP BY company
ORDER BY avg_wue ASC;
"""
pd.read_sql(query_water, conn)

,company,avg_wue,avg_litres_per_M_queries,total_water_billion_litres
0,Google,226.909,1500353.0,12.81
1,AWS,371.304,4044907.0,25.46
2,Microsoft,469.758,6884654.0,20.37
3,Meta,480.234,9196196.0,12.83


## Query 2: Worst PUE Years by Company

In [3]:
query_pue = """
SELECT year, company, 
       ROUND(pue, 3) AS pue, 
       ROUND(energy_waste_pct, 1) AS waste_pct,
       RANK() OVER (PARTITION BY company ORDER BY pue DESC) AS worst_rank
FROM resources
WHERE company != 'All'
ORDER BY pue DESC
LIMIT 20;
"""
pd.read_sql(query_pue, conn)

,year,company,pue,waste_pct,worst_rank
0,2017,AWS,1.305,23.4,1
1,2015,AWS,1.304,23.3,2
2,2016,AWS,1.294,22.7,3
3,2015,Meta,1.251,20.1,1
4,2018,AWS,1.242,19.5,4
5,2015,Microsoft,1.235,19.0,1
6,2017,Meta,1.230,18.7,2
7,2016,Meta,1.226,18.4,3
8,2019,AWS,1.217,17.8,5
9,2016,Microsoft,1.210,17.4,2


## Query 3: Renewable Pledge vs Reality (Greenwash Risk)

In [4]:
query_gap = """
SELECT company, year, 
       renewable_pledged_pct,
       renewable_actual_pct,
       ROUND(renewable_pledged_pct - renewable_actual_pct, 1) AS gap_pct,
       CASE 
           WHEN (renewable_pledged_pct - renewable_actual_pct) > 20 THEN 'High Greenwash Risk'
           WHEN (renewable_pledged_pct - renewable_actual_pct) > 10 THEN 'Moderate Gap'
           ELSE 'On Track'
       END AS status
FROM resources
WHERE company != 'All'
ORDER BY gap_pct DESC
LIMIT 20;
"""
pd.read_sql(query_gap, conn)

,company,year,renewable_pledged_pct,renewable_actual_pct,gap_pct,status
0,AWS,2022,100,62,38.0,High Greenwash Risk
1,Microsoft,2021,100,68,32.0,High Greenwash Risk
2,AWS,2023,100,68,32.0,High Greenwash Risk
3,AWS,2021,85,56,29.0,High Greenwash Risk
4,Microsoft,2022,100,72,28.0,High Greenwash Risk
5,AWS,2020,76,50,26.0,High Greenwash Risk
6,Meta,2020,100,75,25.0,High Greenwash Risk
7,AWS,2024,100,75,25.0,High Greenwash Risk
8,Microsoft,2023,100,78,22.0,High Greenwash Risk
9,Meta,2019,86,65,21.0,High Greenwash Risk


## Query 4: Year-over-Year Electricity Growth Rate

In [5]:
query_yoy = """
SELECT company, year, 
       ROUND(electricity_gwh, 1) AS gwh,
       ROUND(
           (electricity_gwh - LAG(electricity_gwh) OVER (PARTITION BY company ORDER BY year))
           / LAG(electricity_gwh) OVER (PARTITION BY company ORDER BY year) * 100
       , 1) AS yoy_growth_pct
FROM resources
WHERE company != 'All'
ORDER BY company, year;
"""
pd.read_sql(query_yoy, conn)

,company,year,gwh,yoy_growth_pct
0,AWS,2015,8.0,NaN
1,AWS,2016,9.5,17.5
2,AWS,2017,11.3,19.7
3,AWS,2018,13.2,16.8
4,AWS,2019,15.8,19.8
5,AWS,2020,18.4,16.0
6,AWS,2021,20.3,10.4
7,AWS,2022,26.8,31.9
8,AWS,2023,32.0,19.6
9,AWS,2024,37.5,17.2


## Query 5: Cumulative CO2 Emissions by Company

In [6]:
query_co2 = """
SELECT company,
       ROUND(SUM(co2_tons) / 1e6, 2) AS total_co2_million_tonnes,
       ROUND(AVG(co2_per_gwh), 0) AS avg_co2_per_gwh,
       ROUND(MIN(co2_per_gwh), 0) AS best_co2_per_gwh,
       ROUND(MAX(co2_per_gwh), 0) AS worst_co2_per_gwh
FROM resources
WHERE company != 'All'
GROUP BY company
ORDER BY total_co2_million_tonnes DESC;
"""
pd.read_sql(query_co2, conn)

,company,total_co2_million_tonnes,avg_co2_per_gwh,best_co2_per_gwh,worst_co2_per_gwh
0,AWS,44.52,309539.0,88415.0,650743.0
1,Microsoft,30.92,331791.0,117413.0,726228.0
2,Google,22.88,190472.0,64404.0,393940.0
3,Meta,11.06,249654.0,56990.0,642192.0


## Query 6: Best Performing Company-Year Combinations

In [7]:
query_best = """
SELECT company, year,
       ROUND(pue, 3) AS pue,
       ROUND(wue, 3) AS wue,
       ROUND(renewable_actual_pct, 1) AS renewable_pct,
       ROUND(energy_waste_pct, 1) AS waste_pct,
       ROUND(pue * (1 - renewable_actual_pct/100) * wue, 4) AS composite_waste_score
FROM resources
WHERE company != 'All' AND year >= 2020
ORDER BY composite_waste_score ASC
LIMIT 10;
"""
pd.read_sql(query_best, conn)

,company,year,pue,wue,renewable_pct,waste_pct,composite_waste_score
0,Google,2022,1.108,212.060,82.0,9.7,234.9625
1,Google,2020,1.117,224.473,67.0,10.5,250.7363
2,Google,2021,1.108,226.629,76.0,9.7,251.1049
3,Google,2023,1.099,232.565,85.0,9.0,255.5889
4,Google,2024,1.108,238.816,90.0,9.7,264.6081
5,AWS,2024,1.118,322.799,75.0,10.6,360.8893
6,AWS,2023,1.177,334.422,68.0,15.0,393.6147
7,AWS,2022,1.179,361.308,62.0,15.2,425.9821
8,Meta,2024,1.092,407.228,94.0,8.4,444.6930
9,Microsoft,2024,1.110,435.361,83.0,9.9,483.2507


In [8]:
conn.close()
print("Database saved: data/datacenter.db")

Database saved: data/datacenter.db
